<a href="https://colab.research.google.com/github/huutien113/games/blob/main/mp3_to_midi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MP3 piano → MIDI sạch (2 tay)
1. Menu **Runtime → Change runtime type → T4 GPU** (không bắt buộc, nhưng nhanh hơn nhiều)
2. Bấm ▶ ở cell bên dưới, chọn file mp3 khi hộp upload hiện ra
3. File `<tên>_clean.mid` sẽ tự tải về

Muốn chỉnh tham số: sửa dòng `OVERRIDES` ở đầu cell, đặt `MP3_PATH = "tên_file.mp3"` rồi chạy lại.

In [ ]:
import os
import sys
import subprocess
from collections import defaultdict
import numpy as np

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "piano_transcription_inference", "pretty_midi", "librosa"
], check=True)

MP3_PATH = None
REUSE_RAW = True

CFG = {
    "min_vel": 12,
    "min_vel_low": 22,
    "min_dur": 0.04,
    "short_dur": 0.08,
    "weak_ratio": 0.35,
    "ctx_win": 0.5,
    "group_win": 0.04,
    "ghost_vel_ratio": 0.35,
    "ghost_max_dur": 0.10,
    "nb_win": 0.03,
    "nb_vel_ratio": 0.40,
    "hand_span": 14,
    "release_range": 12,
    "max_dur_low": 1.4,
    "max_dur_high": 0.9,
    "max_sustained_lh": 2,
    "max_sustained_rh": 2
}



class NoteItem:
    def __init__(self, start, end, pitch, vel):
        self.start = float(start)
        self.end = float(end)
        self.pitch = int(pitch)
        self.vel = int(vel)

    def get_dur(self):
        return self.end - self.start


def get_start(note):
    return note.start


def get_pitch(note):
    return note.pitch


def get_start_and_pitch(note):
    return (note.start, note.pitch)


def group_by_onset(notes, win):
    sorted_notes = sorted(notes, key=get_start_and_pitch)
    groups = []
    cur = []
    for n in sorted_notes:
        if (len(cur) > 0) == True:
            if (n.start - cur[0].start > win) == True:
                groups.append(cur)
                cur = []
        cur.append(n)
    if (len(cur) > 0) == True:
        groups.append(cur)
    return groups


def remove_short(notes, c):
    if (len(notes) == 0) == True:
        return notes
    sorted_notes = sorted(notes, key=get_start)
    starts_list = []
    vels_list = []
    for n in sorted_notes:
        starts_list.append(n.start)
        vels_list.append(float(n.vel))
    starts = np.array(starts_list)
    vels = np.array(vels_list)

    out = []
    for n in sorted_notes:
        if (n.pitch < 55) == True:
            v_floor = c["min_vel_low"]
        else:
            v_floor = c["min_vel"]

        if (n.vel < v_floor) == True:
            continue
        if (n.get_dur() < c["min_dur"]) == True:
            continue

        if (n.get_dur() < c["short_dur"]) == True:
            if (n.pitch < 72) == True:
                lo = np.searchsorted(starts, n.start - c["ctx_win"], "left")
                hi = np.searchsorted(starts, n.start + c["ctx_win"], "right")
                local_median = float(np.median(vels[lo:hi]))
                if (n.vel < c["weak_ratio"] * local_median) == True:
                    continue
        out.append(n)
    return out


def remove_harmonic_ghosts(notes, c):
    if (len(notes) == 0) == True:
        return notes
    groups = group_by_onset(notes, c["group_win"])
    kept = []
    harmonic_intervals = [12, 19, 24]
    for g in groups:
        by_pitch = {}
        for n in g:
            by_pitch[n.pitch] = n
        for n in g:
            is_ghost = False
            if (n.get_dur() < c["ghost_max_dur"]) == True:
                for k in harmonic_intervals:
                    base_pitch = n.pitch - k
                    if (base_pitch in by_pitch) == True:
                        base_note = by_pitch[base_pitch]
                        if (n.vel < c["ghost_vel_ratio"] * base_note.vel) == True:
                            is_ghost = True
            if is_ghost == False:
                kept.append(n)
    return kept


def kill_neighbors(notes, c):
    sorted_notes = sorted(notes, key=get_start)
    dead = set()
    total = len(sorted_notes)
    for i in range(total):
        a = sorted_notes[i]
        if (id(a) in dead) == True:
            continue
        for j in range(i + 1, total):
            b = sorted_notes[j]
            if (b.start - a.start > c["nb_win"]) == True:
                break
            if (id(b) in dead) == True:
                continue
            diff = abs(a.pitch - b.pitch)
            if (diff == 1) == True:
                if (a.vel < b.vel) == True:
                    weak = a
                    strong = b
                else:
                    weak = b
                    strong = a
                if (weak.vel < c["nb_vel_ratio"] * strong.vel) == True:
                    dead.add(id(weak))
                    if (id(a) in dead) == True:
                        break
    out = []
    for n in sorted_notes:
        if (id(n) in dead) == False:
            out.append(n)
    return out


def fix_same_pitch(notes):
    by_pitch = defaultdict(list)
    for n in notes:
        by_pitch[n.pitch].append(n)
    kept = []
    for pitch in by_pitch:
        lst = by_pitch[pitch]
        lst.sort(key=get_start)
        cur = lst[0]
        for idx in range(1, len(lst)):
            nx = lst[idx]
            if (nx.start - cur.start < 0.05) == True:
                if (nx.end > cur.end) == True:
                    cur.end = nx.end
                if (nx.vel > cur.vel) == True:
                    cur.vel = nx.vel
                continue
            if (nx.start < cur.end) == True:
                new_end = nx.start - 0.005
                min_end = cur.start + 0.03
                if (new_end > min_end) == True:
                    cur.end = new_end
                else:
                    cur.end = min_end
            kept.append(cur)
            cur = nx
        kept.append(cur)
    return kept


def calc_hand_cost(hand, pos, is_left, hand_span):
    if (len(hand) == 0) == True:
        return 0.0
    pitches = []
    for n in hand:
        pitches.append(n.pitch)
    mean_pitch = float(np.mean(pitches))
    cost = abs(mean_pitch - pos)
    span = max(pitches) - min(pitches)
    if (span > hand_span) == True:
        cost = cost + 4.0 * float(span - hand_span)
    if is_left == True:
        if (mean_pitch > 64.0) == True:
            cost = cost + 0.5 * (mean_pitch - 64.0)
    else:
        if (mean_pitch < 60.0) == True:
            cost = cost + 0.5 * (60.0 - mean_pitch)
    return cost


def split_hands(notes, c):
    lh_pos = 48.0
    rh_pos = 72.0
    left_notes = []
    right_notes = []
    groups = group_by_onset(notes, c["group_win"])
    for g in groups:
        ps = sorted(g, key=get_pitch)
        best_cost = None
        best_k = 0
        for k in range(len(ps) + 1):
            lo = ps[:k]
            hi = ps[k:]
            cost_lo = calc_hand_cost(lo, lh_pos, True, c["hand_span"])
            cost_hi = calc_hand_cost(hi, rh_pos, False, c["hand_span"])
            total_cost = cost_lo + cost_hi
            if (best_cost is None) == True:
                best_cost = total_cost
                best_k = k
            elif (total_cost < best_cost) == True:
                best_cost = total_cost
                best_k = k
        lo = ps[:best_k]
        hi = ps[best_k:]
        if (len(lo) > 0) == True:
            lo_pitches = []
            for n in lo:
                lo_pitches.append(n.pitch)
            lh_pos = 0.6 * lh_pos + 0.4 * float(np.mean(lo_pitches))
        if (len(hi) > 0) == True:
            hi_pitches = []
            for n in hi:
                hi_pitches.append(n.pitch)
            rh_pos = 0.6 * rh_pos + 0.4 * float(np.mean(hi_pitches))
        for n in lo:
            left_notes.append(n)
        for n in hi:
            right_notes.append(n)
    return {"left": left_notes, "right": right_notes}

def get_start_and_vel(note):
    return (note.start, note.vel)


def shorten_hand(hand_notes, c, max_sustained):
    groups = group_by_onset(hand_notes, c["group_win"])
    total_groups = len(groups)
    rng = c["release_range"]

    for gi in range(total_groups):
        g = groups[gi]
        for n in g:
            for gj in range(gi + 1, total_groups):
                nxt = groups[gj]
                if (nxt[0].start >= n.end) == True:
                    break
                near_starts = []
                for m in nxt:
                    pitch_diff = abs(m.pitch - n.pitch)
                    if (pitch_diff <= rng) == True:
                        near_starts.append(m.start)
                if (len(near_starts) > 0) == True:
                    earliest_near = min(near_starts)
                    min_allowed = n.start + c["min_dur"]
                    if (earliest_near > min_allowed) == True:
                        n.end = earliest_near
                    else:
                        n.end = min_allowed
                    break

            if (n.pitch < 60) == True:
                cap = c["max_dur_low"]
            else:
                cap = c["max_dur_high"]
            max_end = n.start + cap
            if (n.end > max_end) == True:
                n.end = max_end

    for gi in range(total_groups):
        g = groups[gi]
        t = g[0].start
        active_prev = []
        for prev_idx in range(gi):
            prev_g = groups[prev_idx]
            for n in prev_g:
                if (n.end > t + 0.02) == True:
                    active_prev.append(n)
        keep_old = max_sustained - len(g)
        if (keep_old < 0) == True:
            keep_old = 0
        if (len(active_prev) > keep_old) == True:
            active_prev.sort(key=get_start_and_vel)
            cut_count = len(active_prev) - keep_old
            for idx in range(cut_count):
                old_note = active_prev[idx]
                min_allowed = old_note.start + c["min_dur"]
                if (t > min_allowed) == True:
                    old_note.end = t
                else:
                    old_note.end = min_allowed


def process_notes(notes, c):
    step1 = remove_short(notes, c)
    step2 = remove_harmonic_ghosts(step1, c)
    step3 = kill_neighbors(step2, c)
    step4 = fix_same_pitch(step3)
    hands = split_hands(step4, c)
    left_hand = hands["left"]
    right_hand = hands["right"]
    shorten_hand(left_hand, c, c["max_sustained_lh"])
    shorten_hand(right_hand, c, c["max_sustained_rh"])
    for n in left_hand:
        if (n.end < n.start + 0.03) == True:
            n.end = n.start + 0.03
    for n in right_hand:
        if (n.end < n.start + 0.03) == True:
            n.end = n.start + 0.03
    return {"left": left_hand, "right": right_hand}


def clean_midi(in_path, out_path, custom_cfg):
    import pretty_midi
    c = {}
    for k in CFG:
        c[k] = CFG[k]
    for k in custom_cfg:
        c[k] = custom_cfg[k]

    pm = pretty_midi.PrettyMIDI(in_path)
    notes = []
    for inst in pm.instruments:
        if inst.is_drum == False:
            for n in inst.notes:
                notes.append(NoteItem(n.start, n.end, n.pitch, n.velocity))
    print("Ban đầu:", len(notes), "nốt")

    hands = process_notes(notes, c)
    left_hand = hands["left"]
    right_hand = hands["right"]
    total_clean = len(left_hand) + len(right_hand)
    print("Sau khi dọn:", total_clean, "nốt (tay phải", len(right_hand), ", tay trái", len(left_hand), ")")

    tempi = pm.get_tempo_changes()[1]
    if (len(tempi) > 0) == True:
        init_tempo = float(tempi[0])
    else:
        init_tempo = 120.0

    out = pretty_midi.PrettyMIDI(initial_tempo=init_tempo)

    rh_inst = pretty_midi.Instrument(program=0, name="Right Hand")
    for n in sorted(right_hand, key=get_start):
        midi_note = pretty_midi.Note(
            velocity=int(n.vel),
            pitch=int(n.pitch),
            start=float(n.start),
            end=float(n.end)
        )
        rh_inst.notes.append(midi_note)
    out.instruments.append(rh_inst)

    lh_inst = pretty_midi.Instrument(program=0, name="Left Hand")
    for n in sorted(left_hand, key=get_start):
        midi_note = pretty_midi.Note(
            velocity=int(n.vel),
            pitch=int(n.pitch),
            start=float(n.start),
            end=float(n.end)
        )
        lh_inst.notes.append(midi_note)
    out.instruments.append(lh_inst)

    out.write(out_path)
    return out_path


def run(mp3_path, reuse_raw, custom_cfg):
    import librosa
    import torch
    from google.colab import files
    from piano_transcription_inference import PianoTranscription, sample_rate

    if (mp3_path is None) == True:
        up = files.upload()
        keys = list(up.keys())
        if (len(keys) == 0) == True:
            raise ValueError("Chưa tải file MP3 lên.")
        mp3_path = keys[0]

    base_name = os.path.basename(mp3_path)
    split_name = os.path.splitext(base_name)
    stem = split_name[0]
    raw_path = stem + "_raw.mid"
    out_path = stem + "_clean.mid"

    use_cached = False
    if reuse_raw == True:
        if os.path.exists(raw_path) == True:
            use_cached = True

    if use_cached == True:
        print("Dùng lại", raw_path)
    else:
        load_res = librosa.load(mp3_path, sr=sample_rate, mono=True)
        audio = load_res[0]
        if torch.cuda.is_available() == True:
            device = "cuda"
        else:
            device = "cpu"
        transcriber = PianoTranscription(device=device)
        transcriber.transcribe(audio, raw_path)

    clean_midi(raw_path, out_path, custom_cfg)
    files.download(out_path)


OVERRIDES = {}
run(MP3_PATH, REUSE_RAW, OVERRIDES)

Saving audio.mp3 to audio.mp3
